# PathSnap — Notebook 02: Bayesian optimisation, HMM/Viterbi route reconstruction and export (Phase 2)
**Colab:** Runtime → Run all, upload `pathsnap_colab_bundle.zip` when asked. Partitions: train (80 trips) for fitting, CV and Optuna; student_test (10) for comparison and decoder tuning. Evaluator trips are never used here.

In [ ]:
import os, sys, zipfile, time, json, warnings
warnings.filterwarnings('ignore')
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip -q install shapely pyproj folium lightgbm optuna
    if not os.path.isdir('/content/PathSnap/src'):
        from google.colab import files
        up = files.upload()
        zipfile.ZipFile(list(up)[0]).extractall('/content/PathSnap')
    BASE = '/content/PathSnap'
else:
    BASE = os.environ.get('PATHSNAP_BASE', os.path.abspath('..'))
DATA = os.environ.get('PATHSNAP_DATA', os.path.join(BASE, 'Student folder'))
sys.path.insert(0, os.path.join(BASE, 'src'))
RES = os.path.join(BASE, 'results'); PLOTS = os.path.join(RES, 'plots'); MAPS = os.path.join(RES, 'maps'); ART = os.path.join(BASE, 'artifacts')
for d in (RES, PLOTS, MAPS, ART): os.makedirs(d, exist_ok=True)
print('BASE', BASE, '| DATA', DATA)

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, lightgbm as lgb, optuna
from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score, log_loss
from pathsnap.io import list_trips, load_trip
from pathsnap.pipeline import prepare_trip, DEFAULT_CFG
from pathsnap.features import FEATURES
from pathsnap.hmm import HMMParams, precompute_steps, viterbi, classifier_top1, per_point, reconstruct_route
from pathsnap.metrics import pointwise_accuracy, route_continuity, route_prf
from pathsnap.viz import trip_map
SEED = 42; np.random.seed(SEED); NJ = 4
TRAIN = os.path.join(DATA, 'train'); STEST = os.path.join(DATA, 'student_test')
train_ids, stest_ids = list_trips(TRAIN), list_trips(STEST)
t0 = time.time(); prepared = {}
for part, root, ids in [('train', TRAIN, train_ids), ('student_test', STEST, stest_ids)]:
    for tid in ids: prepared[(part, tid)] = prepare_trip(load_trip(root, tid))
def frame(part):
    out = []
    for (p, tid), pt in prepared.items():
        if p == part:
            d = pt.df.copy(); d['gpt'] = tid + '_' + d['pt'].astype(str); out.append(d)
    return pd.concat(out, ignore_index=True)
TR, TE = frame('train'), frame('student_test')
print('features rebuilt in %.0fs | train rows %d | student-test rows %d | %d features' % (time.time() - t0, len(TR), len(TE), len(FEATURES)))

In [ ]:
def sample_points(df, n, seed=SEED):
    g = df.gpt.unique(); rng = np.random.default_rng(seed)
    keep = set(rng.choice(g, size=min(n, len(g)), replace=False)); return df[df.gpt.isin(keep)]
def top1(df, p):
    d = df[['gpt', 'label']].assign(p=p); return float(d.loc[d.groupby('gpt')['p'].idxmax(), 'label'].mean())
def metr(df, p):
    return dict(macro_f1=f1_score(df.label, (p >= 0.5).astype(int), average='macro'), log_loss=log_loss(df.label, np.clip(p, 1e-7, 1 - 1e-7)), top1=top1(df, p))
POS_W = (1 - TR.label.mean()) / TR.label.mean()
UNTUNED = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1, colsample_bytree=0.8, scale_pos_weight=POS_W)
N_FIT, N_VAL = 20000, 8000
FOLDS = list(GroupKFold(n_splits=5).split(TR, groups=TR.trip))
FOLD_DATA = [(sample_points(TR.iloc[a], N_FIT, SEED + i), sample_points(TR.iloc[b], N_VAL, SEED + i)) for i, (a, b) in enumerate(FOLDS)]
def cv_eval(params):
    rows = []
    for ftr, fva in FOLD_DATA:
        m = lgb.LGBMClassifier(**params, random_state=SEED, verbose=-1, n_jobs=NJ).fit(ftr[FEATURES], ftr.label)
        rows.append(metr(fva, m.predict_proba(fva[FEATURES])[:, 1]))
    return pd.DataFrame(rows)
cv_untuned = cv_eval(UNTUNED); print('untuned CV:', cv_untuned.mean().round(4).to_dict())

## 1. Bayesian hyperparameter optimisation (Optuna TPE)
**Objective:** mean 5-fold GroupKFold (by trip) observation-level Top-1 accuracy on the 80 training trips (same sampled folds for every trial). Macro F1 and log loss are recorded for every trial. TPE builds a probabilistic model of good vs bad regions from past trials and samples where improvement is most likely.

In [ ]:
def objective(trial):
    params = dict(
        num_leaves=trial.suggest_int('num_leaves', 15, 255, log=True),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        n_estimators=trial.suggest_int('n_estimators', 150, 900, step=50),
        min_child_samples=trial.suggest_int('min_child_samples', 5, 200, log=True),
        subsample=trial.suggest_float('subsample', 0.5, 1.0), subsample_freq=1,
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.4, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 30.0, log=True),
        scale_pos_weight=POS_W * trial.suggest_float('pos_weight_factor', 0.1, 1.5))
    r = cv_eval(params)
    trial.set_user_attr('macro_f1', float(r.macro_f1.mean())); trial.set_user_attr('log_loss', float(r.log_loss.mean())); trial.set_user_attr('top1_std', float(r.top1.std()))
    return float(r.top1.mean())
N_TRIALS = 30
optuna.logging.set_verbosity(optuna.logging.WARNING)
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=SEED))
t0 = time.time(); study.optimize(objective, n_trials=N_TRIALS); opt_time = time.time() - t0
trials = study.trials_dataframe(); trials.to_csv(os.path.join(RES, 'optuna_trials.csv'), index=False)
print('Optuna: %d trials in %.1f min | best CV Top-1 %.4f (untuned %.4f)' % (N_TRIALS, opt_time / 60, study.best_value, cv_untuned.top1.mean()))
print('best params:', study.best_params)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
v = trials['value'].values; ax[0].plot(v, 'o', alpha=.6, label='trial'); ax[0].plot(np.maximum.accumulate(v), 'r-', label='best so far')
ax[0].axhline(cv_untuned.top1.mean(), color='k', ls='--', label='untuned'); ax[0].set_xlabel('trial'); ax[0].set_ylabel('CV Top-1'); ax[0].legend(); ax[0].set_title('Optuna optimisation history')
imp = optuna.importance.get_param_importances(study); ax[1].barh(list(imp)[::-1], list(imp.values())[::-1]); ax[1].set_title('Hyperparameter importance (fANOVA)')
plt.tight_layout(); plt.savefig(os.path.join(PLOTS, 'optuna_history_importance.png'), dpi=120); plt.show()

In [ ]:
bp = dict(study.best_params); f = bp.pop('pos_weight_factor'); TUNED = dict(bp, subsample_freq=1, scale_pos_weight=POS_W * f)
cv_tuned = cv_eval(TUNED)
rows = []; models = {}
for name, prm, cvr in [('LightGBM untuned', UNTUNED, cv_untuned), ('LightGBM tuned', TUNED, cv_tuned)]:
    t0 = time.time(); m = lgb.LGBMClassifier(**prm, random_state=SEED, verbose=-1, n_jobs=NJ).fit(TR[FEATURES], TR.label); ft = time.time() - t0
    t0 = time.time(); p = m.predict_proba(TE[FEATURES])[:, 1]; it = time.time() - t0
    s = metr(TE, p); models[name] = m
    rows.append(dict(model=name, cv_macro_f1=cvr.macro_f1.mean(), cv_top1=cvr.top1.mean(), cv_log_loss=cvr.log_loss.mean(), test_top1=s['top1'], test_macro_f1=s['macro_f1'], test_log_loss=s['log_loss'], fit_s=ft, infer_s=it))
tvu = pd.DataFrame(rows); tvu.to_csv(os.path.join(RES, 'tuned_vs_untuned.csv'), index=False); tvu.round(4)

## 2. HMM / Viterbi decoder tuning
Hidden states = candidate arcs per GPS point. Emission = α·log(normalised classifier probability, floored at ε). Transition = −|network distance − straight GPS distance| / (β·max(1,Δt)), minus a U-turn penalty when jumping to the reverse arc; unreachable pairs get a fixed penalty; chain restarts after a 60 s gap.
Tuning data: 16 **training** trips (scored with a model fitted on the other 64 trips, so probabilities are out-of-sample) + the 10 student-test trips. Evaluator trips are not used.

In [ ]:
best = models['LightGBM tuned']
hold = train_ids[::5]
m_hold = lgb.LGBMClassifier(**TUNED, random_state=SEED, verbose=-1, n_jobs=NJ).fit(TR[~TR.trip.isin(hold)][FEATURES], TR[~TR.trip.isin(hold)].label)
dec_trips = []
t0 = time.time()
for part, ids, mdl in [('train', hold, m_hold), ('student_test', stest_ids, best)]:
    for tid in ids:
        pt = prepared[(part, tid)]
        st = precompute_steps(pt.df, pt.pxy, pt.times, pt.net)
        dec_trips.append((part, tid, pt, st, mdl.predict_proba(pt.df[FEATURES])[:, 1]))
print('precomputed route distances for %d trips in %.0fs' % (len(dec_trips), time.time() - t0))
grid = []
for a in [0.3, 0.5, 1.0, 2.0]:
    for b in [2, 3, 5, 10]:
        for u in [0, 3, 8, 15]:
            prm = HMMParams(alpha=a, beta=b, uturn=u); r = dict(alpha=a, beta=b, uturn=u)
            for part in ['train', 'student_test']:
                acc = [(pointwise_accuracy(per_point(st, viterbi(st, p, prm), len(pt.pxy)), pt.ref), len(pt.pxy)) for pp, tid, pt, st, p in dec_trips if pp == part]
                r[f'{part}_acc'] = np.average([x for x, _ in acc], weights=[n for _, n in acc])
            r['combined'] = (r['train_acc'] + r['student_test_acc']) / 2; grid.append(r)
hg = pd.DataFrame(grid).sort_values('combined', ascending=False); hg.to_csv(os.path.join(RES, 'hmm_tuning.csv'), index=False)
BEST_HMM = HMMParams(alpha=float(hg.iloc[0].alpha), beta=float(hg.iloc[0].beta), uturn=float(hg.iloc[0].uturn))
print('best decoder params:', BEST_HMM); hg.head(10).round(4)

## 3. Classifier-only vs classifier + HMM (student-test)

In [ ]:
rows = []; decoded = {}
for pp, tid, pt, st, p in dec_trips:
    if pp != 'student_test': continue
    trt = load_trip(STEST, tid)
    c = per_point(st, classifier_top1(st, p), len(pt.pxy)); h = per_point(st, viterbi(st, p, BEST_HMM), len(pt.pxy))
    rc, rh = reconstruct_route(c, pt.net, fill=False), reconstruct_route(h, pt.net)
    decoded[tid] = (c, h, rc, rh)
    pc, ph = route_prf(rc, trt.route, pt.net), route_prf(rh, trt.route, pt.net)
    rows.append(dict(trip=tid, points=len(pt.pxy), clf_acc=pointwise_accuracy(c, pt.ref), hmm_acc=pointwise_accuracy(h, pt.ref),
        clf_continuity=route_continuity(c, pt.net), hmm_continuity=route_continuity(h, pt.net),
        clf_route_f1=pc['route_f1'], hmm_route_precision=ph['route_precision'], hmm_route_recall=ph['route_recall'], hmm_route_f1=ph['route_f1']))
hc = pd.DataFrame(rows); w = hc.points / hc.points.sum()
summary = hc.drop(columns=['trip', 'points']).mul(w, axis=0).sum(); hc.to_csv(os.path.join(RES, 'hmm_comparison.csv'), index=False)
print('point-weighted averages:'); print(summary.round(4).to_string()); hc.round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4)); x = np.arange(len(hc))
for a_, (c1, c2, t) in zip(ax, [('clf_acc', 'hmm_acc', 'Pointwise accuracy'), ('clf_continuity', 'hmm_continuity', 'Route continuity'), ('clf_route_f1', 'hmm_route_f1', 'Route F1 (length-weighted)')]):
    a_.bar(x - .2, hc[c1], .4, label='classifier only', color='#b0a99f'); a_.bar(x + .2, hc[c2], .4, label='classifier + HMM', color='#8c2f39')
    a_.set_xticks(x); a_.set_xticklabels(hc.trip.str[-3:], rotation=0); a_.set_ylim(max(0, min(hc[c1].min(), hc[c2].min()) - .05), 1.0); a_.set_title(t); a_.set_xlabel('student-test trip')
ax[0].legend(); plt.tight_layout(); plt.savefig(os.path.join(PLOTS, 'hmm_comparison.png'), dpi=120); plt.show()
worse = hc[hc.hmm_acc < hc.clf_acc]; print('Trips where HMM decoding lowered pointwise accuracy:'); print(worse[['trip', 'clf_acc', 'hmm_acc']].round(4))

**Where the HMM did not help.** On trips listed above (e.g. trip 7 in our runs) the vehicle ends the trip very slowly / manoeuvring: GPS heading is unreliable and consecutive points barely move, so the transition model cannot separate the two directions of the same road and the decoder sometimes keeps the reverse arc for a stretch. The HMM still guarantees a connected route (continuity ≈ 1.0), which is why route F1 stays high even there. Remaining errors elsewhere are mostly at junctions (the predicted arc touches the reference arc) where the derived per-point label is itself ambiguous.

In [ ]:
for tid in stest_ids[:2] if '00000077' not in stest_ids else ['00000077', stest_ids[0]]:
    pt = prepared[('student_test', tid)]; trt = load_trip(STEST, tid); c, h, rc, rh = decoded[tid]
    m = trip_map(pt.track, pt.net, ref_route=trt.route, routes={'Classifier-only (LightGBM tuned)': rc, 'Classifier + HMM/Viterbi': rh})
    m.save(os.path.join(MAPS, f'hmm_vs_classifier_{tid}.html'))
m

## 4. Export artifacts for the API (top-3 models: tuned LightGBM, Random Forest, MLP)

In [ ]:
import joblib, platform, sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
fit_df = sample_points(TR, 50000, SEED)
rf = RandomForestClassifier(n_estimators=150, max_depth=14, min_samples_leaf=5, class_weight='balanced_subsample', n_jobs=NJ, random_state=SEED).fit(fit_df[FEATURES], fit_df.label)
mlp = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64, 32), alpha=1e-4, batch_size=512, max_iter=40, early_stopping=True, random_state=SEED)).fit(fit_df[FEATURES], fit_df.label)
best.booster_.save_model(os.path.join(ART, 'lightgbm_tuned.txt')); joblib.dump(best, os.path.join(ART, 'lightgbm_tuned.joblib'))
joblib.dump(rf, os.path.join(ART, 'random_forest.joblib')); joblib.dump(mlp, os.path.join(ART, 'mlp.joblib'))
reg = []
for key, label, fname, typ, colour, mdl in [('lightgbm', 'LightGBM (tuned)', 'lightgbm_tuned.txt', 'lightgbm_native', '#8c2f39', best),
                                            ('random_forest', 'Random Forest', 'random_forest.joblib', 'sklearn', '#1f5f5b', rf),
                                            ('mlp', 'MLP 64-32', 'mlp.joblib', 'sklearn', '#b07d1a', mlp)]:
    p = mdl.predict_proba(TE[FEATURES])[:, 1]; s = metr(TE, p)
    acc = []
    for tid in stest_ids:
        pt = prepared[('student_test', tid)]; st = next(x[3] for x in dec_trips if x[0] == 'student_test' and x[1] == tid)
        acc.append((pointwise_accuracy(per_point(st, viterbi(st, p[(TE.trip == tid).values], BEST_HMM), len(pt.pxy)), pt.ref), len(pt.pxy)))
    reg.append(dict(key=key, label=label, file=fname, type=typ, colour=colour, student_test=dict(top1=round(s['top1'], 4), macro_f1=round(s['macro_f1'], 4), log_loss=round(s['log_loss'], 4),
               hmm_pointwise=round(float(np.average([a for a, _ in acc], weights=[n for _, n in acc])), 4))))
json.dump(reg, open(os.path.join(ART, 'models.json'), 'w'), indent=1)
json.dump({'cfg': DEFAULT_CFG, 'features': FEATURES}, open(os.path.join(ART, 'feature_config.json'), 'w'), indent=1)
json.dump(dict(BEST_HMM.__dict__), open(os.path.join(ART, 'hmm_config.json'), 'w'), indent=1)
json.dump({'python': platform.python_version(), 'sklearn': sklearn.__version__, 'lightgbm': lgb.__version__, 'numpy': np.__version__, 'pandas': pd.__version__, 'optuna': optuna.__version__,
           'tuned_params': {k: (float(v) if isinstance(v, (np.floating, float)) else v) for k, v in TUNED.items()}, 'optuna_trials': N_TRIALS, 'optuna_minutes': opt_time / 60}, open(os.path.join(ART, 'versions.json'), 'w'), indent=1)
pd.DataFrame(reg).drop(columns=['student_test']).join(pd.DataFrame([r['student_test'] for r in reg]))

In [ ]:
if IN_COLAB:
    with zipfile.ZipFile('/content/pathsnap_phase2.zip', 'w', zipfile.ZIP_DEFLATED) as z:
        for sub in ['artifacts', 'results']:
            for r_, _, fs in os.walk(os.path.join(BASE, sub)):
                for f_ in fs:
                    if not f_.endswith('.parquet'): z.write(os.path.join(r_, f_), os.path.relpath(os.path.join(r_, f_), BASE))
    from google.colab import files; files.download('/content/pathsnap_phase2.zip')
print(sorted(os.listdir(ART)))